# 365 Probabilidades — Dia #062
## Qual a probabilidade de ser o mais novo da sala custar um diagnóstico?

**Tipo:** Comportamental
**Data de publicação:** 2026-08-14
**Ferramenta:** Python
**Decisão analisada:** Como interpretar um diagnóstico feito no início da vida escolar?
**Hashtag:** #365Probabilidades #Dia062

---

### 📖 A História

Em quase todo sistema escolar existe uma data de corte. Quem nasce de um lado
dela entra na turma daquele ano. Quem nasce do outro espera doze meses.

Na prática, isso significa que dentro da mesma sala há crianças com quase um ano
inteiro de diferença. Aos seis anos, doze meses são uma eternidade de
desenvolvimento: de atenção, de linguagem, de controle de impulso.

O professor, olhando para a turma, não vê datas de nascimento. Vê comportamento.
E compara cada criança com as outras que estão ali.

A pergunta é o que acontece com a criança que é sempre a mais nova de todas.

---

### 📚 O Conceito: efeito de idade relativa

Idade relativa é a posição de alguém dentro do próprio grupo, não a idade
absoluta. O efeito de idade relativa já foi documentado em esporte de alto
rendimento, em desempenho escolar e em avaliação de professores.

A hipótese aqui é específica: comportamento que é normal para a idade real da
criança pode ser lido como sintoma quando comparado com colegas até onze meses
mais velhos.

Testar isso exige mais do que comparar dois grupos. Exige mostrar que a diferença
aparece **só** onde a regra de corte existe, **só** depois que a escola começa, e
**só** no desfecho que a hipótese prevê.

---

### 🧮 O Modelo

Distribuição Beta com prior uniforme para cada taxa, razão de risco com IC 95%
pelo erro-padrão do logaritmo, e o painel de testes de falsificação do estudo.

**Fonte principal:**
- Layton, Barnett, Hicks & Jena, 2018 — *New England Journal of Medicine* 379(22),
  2122-2130. Estudo observacional com base em dados de sinistro de plano de saúde.
  N=407.846 crianças nascidas entre 2007 e 2009, seguidas até dezembro de 2015.
  Diagnóstico por códigos CID-9. Desenho quase experimental com três conjuntos de
  testes de falsificação pré-especificados.

**Âncoras externas:**
- Frisira, Holland & Sayal, 2024 — *European Child & Adolescent Psychiatry*.
  Revisão sistemática com meta-análise, 32 estudos. RR combinado de 1,38
  [1,36 · 1,52] para diagnóstico e 1,28 [1,21 · 1,36] para medicação. I² = 99%.
- Morrow et al., 2012 — *CMAJ*. N=937.943 crianças canadenses de 6 a 12 anos.
- Hu, Faraone & Morgan, 2026 — *Journal of Attention Disorders*. N=11.410, coorte
  ECLS-K. **Não** encontrou o efeito. Ver limitações.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

np.random.seed(42)

print("✅ Bibliotecas carregadas")


In [ ]:
# --- DADOS DA LITERATURA ---
# Layton, Barnett, Hicks & Jena, 2018 — N Engl J Med 379(22):2122-2130
# Todos os valores lidos direto do artigo. Nada reconstruído.

n_total_estudo = 407846        # crianças em todos os estados, nascidas 2007-2009

# Estados COM corte em 1º de setembro — DIAGNÓSTICO
n_agosto = 36319
casos_agosto = 309
taxa_agosto = 85.1             # por 10.000 · IC 95% [75,6 · 94,2]
ic_agosto = (75.6, 94.2)

n_setembro = 35353
casos_setembro = 225
taxa_setembro = 63.6           # por 10.000 · IC 95% [55,4 · 71,9]
ic_setembro = (55.4, 71.9)

dif_diagnostico = 21.5         # por 10.000 · IC 95% [8,8 · 34,0]
ic_dif_diagnostico = (8.8, 34.0)

# Estados COM corte — TRATAMENTO MEDICAMENTOSO
trat_agosto = 52.9             # por 10.000 · IC 95% [45,4 · 60,3] · 192 de 36.319
casos_trat_agosto = 192
ic_trat_agosto = (45.4, 60.3)

trat_setembro = 40.4           # por 10.000 · IC 95% [33,8 · 47,1] · 143 de 35.353
casos_trat_setembro = 143
ic_trat_setembro = (33.8, 47.1)

dif_tratamento = 12.5          # por 10.000 · IC 95% [2,43 · 22,4]
ic_dif_tratamento = (2.43, 22.4)

# --- TESTES DE FALSIFICAÇÃO (o coração do desenho) ---
# 1. Estados SEM o corte de 1º de setembro
dif_sem_corte = 8.9
ic_sem_corte = (-14.9, 20.8)   # atravessa o zero

# 2. Antes e depois de a escola começar
dif_aos_4_anos = 0.2           # antes de qualquer criança estar na escola
ic_aos_4_anos = (-5.6, 5.9)    # atravessa o zero
dif_aos_7_anos = 17.8          # com todos já na escola
ic_aos_7_anos = (6.8, 28.7)

# 3. Estratificação por sexo
dif_meninos = 32.5
ic_meninos = (11.9, 53.2)
dif_meninas = 10.7
ic_meninas = (-3.2, 24.6)      # atravessa o zero

# 4. Doenças que a escola não explica: asma, obesidade e diabetes
#    não diferiram entre nascidos em agosto e em setembro (Fig. S7 do artigo)

# Entre as crianças medicadas
dias_extras_medicacao = 120    # IC 95% [40 · 200]
ic_dias_extras = (40, 200)

# Fator de correção: NÃO se aplica (ver nota metodológica)
aplica_fator_080 = False

print("=" * 70)
print("  DADOS — LAYTON ET AL., 2018 · N Engl J Med · N=407.846")
print("=" * 70)
print(f"\n  ESTADOS COM CORTE EM 1º DE SETEMBRO — DIAGNÓSTICO")
print(f"  → Nascidos em agosto:    {taxa_agosto:.1f} por 10.000  "
      f"[{ic_agosto[0]:.1f} · {ic_agosto[1]:.1f}]   ({casos_agosto} de {n_agosto:,})")
print(f"  → Nascidos em setembro:  {taxa_setembro:.1f} por 10.000  "
      f"[{ic_setembro[0]:.1f} · {ic_setembro[1]:.1f}]   ({casos_setembro} de {n_setembro:,})")
print(f"  → Diferença absoluta:    {dif_diagnostico:.1f} por 10.000  "
      f"[{ic_dif_diagnostico[0]:.1f} · {ic_dif_diagnostico[1]:.1f}]")
print(f"\n  ESTADOS COM CORTE — TRATAMENTO MEDICAMENTOSO")
print(f"  → Nascidos em agosto:    {trat_agosto:.1f} por 10.000  "
      f"[{ic_trat_agosto[0]:.1f} · {ic_trat_agosto[1]:.1f}]")
print(f"  → Nascidos em setembro:  {trat_setembro:.1f} por 10.000  "
      f"[{ic_trat_setembro[0]:.1f} · {ic_trat_setembro[1]:.1f}]")
print(f"  → Diferença absoluta:    {dif_tratamento:.1f} por 10.000  "
      f"[{ic_dif_tratamento[0]:.2f} · {ic_dif_tratamento[1]:.1f}]")
print(f"\n  TESTES DE FALSIFICAÇÃO (diferença por 10.000, IC 95%)")
print(f"  → Estados sem o corte:   {dif_sem_corte:+.1f}  "
      f"[{ic_sem_corte[0]:.1f} · {ic_sem_corte[1]:.1f}]   ← atravessa o zero")
print(f"  → Aos 4 anos:            {dif_aos_4_anos:+.1f}  "
      f"[{ic_aos_4_anos[0]:.1f} · {ic_aos_4_anos[1]:.1f}]   ← atravessa o zero")
print(f"  → Aos 7 anos:            {dif_aos_7_anos:+.1f}  "
      f"[{ic_aos_7_anos[0]:.1f} · {ic_aos_7_anos[1]:.1f}]")
print(f"  → Meninos:               {dif_meninos:+.1f}  "
      f"[{ic_meninos[0]:.1f} · {ic_meninos[1]:.1f}]")
print(f"  → Meninas:               {dif_meninas:+.1f}  "
      f"[{ic_meninas[0]:.1f} · {ic_meninas[1]:.1f}]   ← atravessa o zero")
print(f"\n  → Asma, obesidade e diabetes: sem diferença entre agosto e setembro")
print(f"\n  Fator ×0.80 aplicado:    {'sim' if aplica_fator_080 else 'não'}")
print("=" * 70)


In [ ]:
# --- O MODELO ---
# Duas peças: (1) posterior Beta de cada taxa, (2) razão de risco com IC 95%.

# --- (1) BAYESIANO: prior uniforme Beta(1,1), conjugada da binomial ---
# Posterior = Beta(casos + 1, n - casos + 1). O prior é declarado, então o
# rótulo "bayesiano" cabe aqui — ao contrário dos dias de simulação.

post_agosto = stats.beta(casos_agosto + 1, n_agosto - casos_agosto + 1)
post_setembro = stats.beta(casos_setembro + 1, n_setembro - casos_setembro + 1)

ic_post_agosto = np.array(post_agosto.interval(0.95)) * 10000
ic_post_setembro = np.array(post_setembro.interval(0.95)) * 10000
media_post_agosto = post_agosto.mean() * 10000
media_post_setembro = post_setembro.mean() * 10000

# Probabilidade posterior de a taxa de agosto ser maior que a de setembro
amostras = 200000
p_agosto_maior = np.mean(post_agosto.rvs(amostras) > post_setembro.rvs(amostras))

# --- (2) FREQUENTISTA: razão de risco com IC 95% pelo erro-padrão do log ---
rr = (casos_agosto / n_agosto) / (casos_setembro / n_setembro)
se_log_rr = np.sqrt(1/casos_agosto + 1/casos_setembro - 1/n_agosto - 1/n_setembro)
ic_rr = (np.exp(np.log(rr) - 1.96 * se_log_rr),
         np.exp(np.log(rr) + 1.96 * se_log_rr))

rr_tratamento = (casos_trat_agosto / n_agosto) / (casos_trat_setembro / n_setembro)
se_log_rr_t = np.sqrt(1/casos_trat_agosto + 1/casos_trat_setembro
                      - 1/n_agosto - 1/n_setembro)
ic_rr_t = (np.exp(np.log(rr_tratamento) - 1.96 * se_log_rr_t),
           np.exp(np.log(rr_tratamento) + 1.96 * se_log_rr_t))

# --- (3) Tradução para escala de sala de aula ---
# Quantas crianças a mais recebem diagnóstico a cada 10.000 nascidas em agosto
crianca_por_turma = 25
turmas_por_diagnostico_extra = 10000 / dif_diagnostico / crianca_por_turma

print("=" * 70)
print("  MODELO — O CUSTO DE SER O MAIS NOVO DA SALA")
print("=" * 70)
print(f"\n  POSTERIOR BETA (prior uniforme Beta(1,1))")
print(f"  → Agosto:    {media_post_agosto:.1f} por 10.000  "
      f"IC 95% [{ic_post_agosto[0]:.1f} · {ic_post_agosto[1]:.1f}]")
print(f"  → Setembro:  {media_post_setembro:.1f} por 10.000  "
      f"IC 95% [{ic_post_setembro[0]:.1f} · {ic_post_setembro[1]:.1f}]")
print(f"  → P(taxa de agosto > taxa de setembro) = {p_agosto_maior*100:.2f}%")
print(f"\n  RAZÃO DE RISCO (IC 95% pelo erro-padrão do log)")
print(f"  → Diagnóstico:  RR = {rr:.2f}  [{ic_rr[0]:.2f} · {ic_rr[1]:.2f}]")
print(f"  → Tratamento:   RR = {rr_tratamento:.2f}  "
      f"[{ic_rr_t[0]:.2f} · {ic_rr_t[1]:.2f}]")
print(f"  → Ou seja: {(rr-1)*100:.0f}% a mais de diagnóstico e "
      f"{(rr_tratamento-1)*100:.0f}% a mais de tratamento")
print(f"\n  O QUE OS TESTES DE FALSIFICAÇÃO MOSTRAM")
print(f"  → Sem a regra de corte, a diferença some:   {dif_sem_corte:+.1f} "
      f"[{ic_sem_corte[0]:.1f} · {ic_sem_corte[1]:.1f}]")
print(f"  → Antes da escola (4 anos), não existe:     {dif_aos_4_anos:+.1f} "
      f"[{ic_aos_4_anos[0]:.1f} · {ic_aos_4_anos[1]:.1f}]")
print(f"  → Depois da escola (7 anos), existe:        {dif_aos_7_anos:+.1f} "
      f"[{ic_aos_7_anos[0]:.1f} · {ic_aos_7_anos[1]:.1f}]")
print(f"\n  ESCALA DE SALA DE AULA")
print(f"  → {dif_diagnostico:.1f} diagnósticos a mais por 10.000 crianças")
print(f"  → cerca de 1 criança a cada {turmas_por_diagnostico_extra:.0f} turmas "
      f"de {crianca_por_turma}")
print(f"  → e quem é medicado toma {dias_extras_medicacao} dias a mais de remédio "
      f"[{ic_dias_extras[0]} · {ic_dias_extras[1]}]")
print("=" * 70)


In [ ]:
# --- VISUALIZAÇÃO — GRÁFICOS SEPARADOS ---

VERMELHO = '#c0392b'
DOURADO = '#c8a84b'
VERDE = '#1a5f5a'
CINZA = '#6b6a64'

# ── GRÁFICO 1 — Taxas de diagnóstico e tratamento ──
fig1, ax1 = plt.subplots(figsize=(12, 8))

posicoes = [0.85, 1.15, 1.85, 2.15]
valores = [taxa_agosto, taxa_setembro, trat_agosto, trat_setembro]
ics = [ic_agosto, ic_setembro, ic_trat_agosto, ic_trat_setembro]
cores = [VERMELHO, VERDE, VERMELHO, VERDE]
rotulos = ['agosto', 'setembro', 'agosto', 'setembro']

for pos, val, ic, cor in zip(posicoes, valores, ics, cores):
    ax1.errorbar(pos, val, yerr=[[val - ic[0]], [ic[1] - val]],
                 fmt='o', color=cor, markersize=14, capsize=10,
                 elinewidth=2.5, capthick=2.5)
    ax1.text(pos, ic[1] + 2.5, f'{val:.1f}'.replace('.', ','),
             ha='center', fontsize=15, fontweight='bold', color=cor)

for pos, rot in zip(posicoes, rotulos):
    ax1.text(pos, 26, rot, ha='center', fontsize=12, color=CINZA)

ax1.set_xticks([1.0, 2.0])
ax1.set_xticklabels(['DIAGNÓSTICO de TDAH', 'TRATAMENTO com medicação'], fontsize=13)
ax1.set_xlim(0.5, 2.5)
ax1.set_ylim(22, 105)
ax1.set_ylabel('Casos por 10.000 crianças')
ax1.set_title('Nascer em agosto, num estado que corta a matrícula em 1º de setembro\n'
              'Layton et al., 2018 — N Engl J Med · 36.319 crianças de agosto, 35.353 de setembro',
              fontsize=13.5, pad=15)

ax1.annotate('', xy=(1.15, taxa_setembro), xytext=(0.85, taxa_agosto),
             arrowprops=dict(arrowstyle='<->', color=DOURADO, lw=2))
ax1.text(1.0, 76, f'+{dif_diagnostico:.1f}'.replace('.', ','),
         ha='center', fontsize=13, color=DOURADO, fontweight='bold')
ax1.annotate('', xy=(2.15, trat_setembro), xytext=(1.85, trat_agosto),
             arrowprops=dict(arrowstyle='<->', color=DOURADO, lw=2))
ax1.text(2.0, 47.5, f'+{dif_tratamento:.1f}'.replace('.', ','),
         ha='center', fontsize=13, color=DOURADO, fontweight='bold')

plt.figtext(0.5, 0.005,
            'Barras: IC 95%. Fonte: Layton, Barnett, Hicks & Jena, 2018 — '
            'N Engl J Med 379(22) | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-062-grafico-01-taxas.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# ── GRÁFICO 2 — O painel de falsificação ──
fig2, ax2 = plt.subplots(figsize=(12, 8.5))

itens = [
    ('Estados COM corte em 1º de setembro',      dif_diagnostico,  ic_dif_diagnostico, True),
    ('Estados SEM esse corte',                   dif_sem_corte,    ic_sem_corte,       False),
    ('Aos 4 anos (antes da escola)',             dif_aos_4_anos,   ic_aos_4_anos,      False),
    ('Aos 7 anos (já na escola)',                dif_aos_7_anos,   ic_aos_7_anos,      True),
    ('Meninos',                                  dif_meninos,      ic_meninos,         True),
    ('Meninas',                                  dif_meninas,      ic_meninas,         False),
    ('Tratamento com medicação',                 dif_tratamento,   ic_dif_tratamento,  True),
]

ys = np.arange(len(itens))[::-1]
for y, (nome, val, ic, significativo) in zip(ys, itens):
    cor = VERMELHO if significativo else CINZA
    ax2.plot([ic[0], ic[1]], [y, y], color=cor, linewidth=3, solid_capstyle='round')
    ax2.plot([ic[0], ic[0]], [y - 0.14, y + 0.14], color=cor, linewidth=2.5)
    ax2.plot([ic[1], ic[1]], [y - 0.14, y + 0.14], color=cor, linewidth=2.5)
    ax2.scatter([val], [y], color=cor, s=130, zorder=5)
    ax2.text(58, y, f'{val:+.1f}'.replace('.', ','), va='center',
             fontsize=12, color=cor, fontweight='bold')

ax2.axvline(x=0, color='#333', linewidth=2)
ax2.text(0, len(itens) - 0.32, 'sem diferença', ha='center',
         fontsize=10, color='#333',
         bbox=dict(boxstyle='round,pad=0.25', facecolor='white', edgecolor='none'))

ax2.set_yticks(ys)
ax2.set_yticklabels([i[0] for i in itens], fontsize=12)
ax2.set_xlim(-20, 68)
ax2.set_ylim(-0.7, len(itens) - 0.15)
ax2.set_xlabel('Diferença na taxa entre nascidos em agosto e em setembro\n'
               '(casos por 10.000 crianças, com IC 95%)')
ax2.set_title('O teste que separa achado de coincidência\n'
              'Em cinza, os intervalos que atravessam o zero: ali a diferença some',
              fontsize=13.5, pad=15)
ax2.grid(axis='y', alpha=0)

plt.figtext(0.5, 0.005,
            'Fonte: Layton et al., 2018 — N Engl J Med 379(22) · asma, obesidade e diabetes '
            'também não diferiram | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-062-grafico-02-falsificacao.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# ── GRÁFICO 3 — Assinatura estatística: posteriores Beta e razão de risco ──
fig3, ax3 = plt.subplots(figsize=(12, 8))

x = np.linspace(40, 115, 2000) / 10000
y_ago = post_agosto.pdf(x)
y_set = post_setembro.pdf(x)

ax3.plot(x * 10000, y_ago, color=VERMELHO, linewidth=3, label='nascidos em agosto')
ax3.fill_between(x * 10000, y_ago, alpha=0.18, color=VERMELHO)
ax3.plot(x * 10000, y_set, color=VERDE, linewidth=3, label='nascidos em setembro')
ax3.fill_between(x * 10000, y_set, alpha=0.18, color=VERDE)

for ic, cor in [(ic_post_agosto, VERMELHO), (ic_post_setembro, VERDE)]:
    ax3.axvline(x=ic[0], color=cor, linestyle='--', linewidth=1.2, alpha=0.7)
    ax3.axvline(x=ic[1], color=cor, linestyle='--', linewidth=1.2, alpha=0.7)

ax3.set_xlabel('Taxa de diagnóstico de TDAH (casos por 10.000 crianças)')
ax3.set_ylabel('Densidade posterior')
ax3.set_title('A assinatura estatística do dia\n'
              'Posterior Beta com prior uniforme · razão de risco com IC 95%',
              fontsize=13.5, pad=15)
ax3.legend(loc='upper left', frameon=False, fontsize=12)

texto = (f'RR = {rr:.2f}  [IC 95%: {ic_rr[0]:.2f} · {ic_rr[1]:.2f}]\n'
         f'P(agosto > setembro) = {p_agosto_maior*100:.2f}%\n'
         f'Diferença: +{dif_diagnostico:.1f} por 10 mil  '
         f'[{ic_dif_diagnostico[0]:.1f} · {ic_dif_diagnostico[1]:.1f}]').replace('.', ',')
ax3.set_ylim(0, max(y_ago.max(), y_set.max()) * 1.42)
ax3.text(0.98, 0.985, texto, transform=ax3.transAxes, ha='right', va='top',
         fontsize=13, color='#333', family='monospace',
         bbox=dict(boxstyle='round,pad=0.7', facecolor='white',
                   edgecolor=DOURADO, linewidth=1.5))

plt.figtext(0.5, 0.005,
            'Posterior Beta(casos+1, n−casos+1). Fonte: Layton et al., 2018 — '
            'N Engl J Med 379(22) | #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-062-grafico-03-assinatura.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")


### 💡 O Insight

**Aos 4 anos, antes de qualquer criança pisar na escola, a diferença entre nascidos
em agosto e em setembro é de 0,2 por 10.000. Aos 7 anos, com todos já na sala de
aula, é de 17,8.**

A diferença não existe antes da escola. Ela nasce lá dentro.

Nos estados que cortam a matrícula em 1º de setembro, quem nasce em agosto recebe
diagnóstico de TDAH a uma taxa de **85,1 por 10.000**, contra **63,6** de quem
nasce em setembro. São **34% a mais**, com razão de risco de 1,34. Em medicação, a
diferença se repete: 52,9 contra 40,4. E entre as crianças que são medicadas, as de
agosto tomam em média **120 dias a mais** de remédio.

Nos estados que não usam essa data de corte, a diferença desaparece.

Nada disso é sobre agosto. É sobre ser o mais novo de uma sala em que todo mundo
é comparado com todo mundo, num momento da vida em que onze meses separam quem
consegue ficar sentado de quem ainda não consegue.

O comportamento da criança de agosto pode estar perfeitamente dentro do esperado
para a idade dela. O problema é que a régua não é a idade dela. É a turma.

A pergunta que fica:

*Quantas das coisas que você acha que são um problema seu são só a régua errada?*

---

### 🔬 Nota Metodológica

**O fator de correção ×0.80 não foi aplicado.** Ele desconta o otimismo de
proporções vindas de survey populacional autorrelatado. Aqui os dados são
administrativos: códigos de diagnóstico e registros de prescrição em base de
sinistro de plano de saúde. Não há autorrelato no desfecho.

**O rótulo bayesiano cabe neste dia.** O prior é declarado e é uniforme,
Beta(1,1), conjugada da binomial. O posterior é Beta(casos+1, n−casos+1). A razão
de risco, ao lado, é frequentista, com IC 95% pelo erro-padrão do logaritmo. Os
dois métodos aparecem juntos de propósito, e cada um está nomeado corretamente.

**Sobre linguagem causal.** O estudo é observacional, mas o desenho é quase
experimental: a data de corte funciona como uma divisão quase arbitrária entre
crianças que, em tudo o mais, são parecidas. Os três testes de falsificação
pré-especificados (estados sem o corte, comparação antes e depois do início da
escola, e desfechos que a escola não deveria afetar) sustentam uma leitura mais
forte que a de simples associação. Ainda assim, não é um ensaio randomizado.

---

### ⚠️ Limitações do Modelo

- **Existe um estudo recente que não encontra o efeito.** Hu, Faraone & Morgan,
  2026, *Journal of Attention Disorders*, coorte ECLS-K com N=11.410 acompanhada
  do jardim de infância ao quinto ano, não observou o efeito de idade relativa.
  Os autores sugerem que a prática diagnóstica pode ter mudado desde as bases mais
  antigas. A evidência não é unânime
- **Os estudos discordam sobre sexo.** Em Layton, o efeito é significativo em
  meninos (+32,5 [11,9 · 53,2]) e não significativo em meninas (+10,7 [−3,2 · 24,6]).
  Em Morrow et al., 2012 (*CMAJ*, N=937.943), o risco relativo era maior nas meninas.
  Não trate nenhuma das duas versões como fato estabelecido
- **A meta-análise tem heterogeneidade altíssima.** Frisira, Holland & Sayal, 2024,
  32 estudos, RR 1,38 [1,36 · 1,52], mas com I² = 99%. O efeito médio existe; a
  variação entre contextos é enorme
- Dados de sinistro medem **diagnóstico registrado e prescrição preenchida**, não
  prevalência real do transtorno nem gravidade
- A população é de crianças com plano de saúde privado nos Estados Unidos, com
  regras escolares próprias. Não é amostra representativa do país nem transferível
  direto para o Brasil
- Diagnóstico codificado em CID-9, no período de 2007 a 2015
- O modelo Beta assume independência entre as crianças dentro de cada grupo, o que
  ignora agrupamento por estado, por escola e por médico

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*
